# Notebook de adquisición — Sectores críticos de siniestralidad vial

**Alcance aplicado:** por la Ficha, el proyecto **excluye el año en curso y las cifras preliminares**. Por eso aquí se filtra la Fuente 2 a **años cerrados (2015–2023)**, dejando fuera 2024 y 2025.

**Columna en común para el cruce:** código DANE del municipio (`divipola` en Fuente 1 = `CodigoMunicipioHecho` en Fuente 2). La Fuente 1 no trae año, así que el cruce es a nivel de municipio (el año queda solo del lado de la Fuente 2, agregado).

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

## 1. Lectura de las fuentes

In [2]:
df_sectores = pd.read_csv('rs3u-8r4q.csv', sep=',', encoding='utf-8-sig')
print("Shape sectores críticos:", df_sectores.shape)
df_sectores.head()

Shape sectores críticos: (316, 13)


,id_mt,entidad,gizscore,fallecidos,gipvalue,tramo,nombre,latitud,longitud,pr,municipio,departamento,divipola
0,4G013,ANI,1.771262,11,0.076517,Bogotá - Villavicencio,IP – Chirajara - Fundadores,4.452987,-74.046677,10,CHIPAQUE,CUNDINAMARCA,25178
1,4G013,ANI,3.818008,23,0.000135,Bogotá - Villavicencio,IP – Chirajara - Fundadores,4.228251,-73.818342,58,GUAYABETAL,CUNDINAMARCA,25335
2,4G013,ANI,2.796662,1,0.005163,Bogotá - Villavicencio,IP – Chirajara - Fundadores,4.203601,-73.807188,60,GUAYABETAL,CUNDINAMARCA,25335
3,4G013,ANI,1.725132,1,0.084504,Bogotá - Villavicencio,IP – Chirajara - Fundadores,4.121453,-73.643971,85,VILLAVICENCIO,META,50001
4,<Null>,INVIAS-OTROS,1.725132,13,0.084504,Granada - Villavicencio,NaN,4.115757,-73.647497,sd,VILLAVICENCIO,META,50001


In [3]:
df_historico = pd.read_csv('02_Comportamiento_datos_históricos.csv', sep=';', encoding='utf-8-sig')
print("Shape comportamiento histórico (sin filtrar):", df_historico.shape)
df_historico.head()

Shape comportamiento histórico (sin filtrar): (77888, 15)


,AnoHecho,CodigoMunicipioHecho,Departamento,Municipio,Zona,Sexo,Edad,UsuarioVia,ActorVial,TipoVehiculoGrupo,ObjetoColision,CondicionVictima,DiaOcurrencia,Rango3horas,ConteoRegistros
0,2023,25183,Cundinamarca,Choconta,Urbana,Mujer,22.0,Usuario de V.Individual,Usuario de vehículo,Transporte Individual,Transporte individual,Pasajero,6.Sábado,00:00 a 02:59,1
1,2017,11001,Bogota Dc,Bogota,Urbana,Hombre,73.0,Peatón,Peatón,No Aplica,Motocicleta,Peatón,1.Lunes,NaN,1
2,2017,41298,Huila,Garzon,Urbana,Mujer,4.0,Peatón,Peatón,No Aplica,Transporte individual,Peatón,4.Jueves,NaN,1
3,2019,76834,Valle Del Cauca,Tulua,Urbana,Mujer,76.0,Peatón,Peatón,No Aplica,Sin info.,Peatón,5.Viernes,NaN,1
4,2016,50001,Meta,Villavicencio,Urbana,Hombre,21.0,Usuario de moto,Usuario de moto,Motocicleta,Transporte individual,Conductor,5.Viernes,18:00 a 20:59,1


## 2. Aplicar el alcance: solo años cerrados

La Ficha excluye explícitamente "el año en curso y los meses con cifras preliminares aún sujetas a revisión". Se filtra la Fuente 2 a 2015–2023 (se descartan 2024 y 2025).

In [4]:
anos_cerrados = [a for a in range(2015, 2024)]
df_historico = df_historico[df_historico['AnoHecho'].isin(anos_cerrados)].copy()
print("Shape histórico filtrado a años cerrados:", df_historico.shape)
print("Años incluidos:", sorted(df_historico.AnoHecho.unique()))

Shape histórico filtrado a años cerrados: (64042, 15)
Años incluidos: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


## 3. Por qué se agrega antes de cruzar

Igual que con las otras fuentes del Observatorio, ninguna fila de `df_historico` es un accidente único: son conteos por combinación de categorías. Se agrega por municipio (código DANE) sumando `ConteoRegistros`.

`df_sectores`, en cambio, sí tiene una fila por tramo vial — pero un mismo municipio puede tener varios tramos críticos, así que también se agrega por municipio antes del cruce (para comparar "fallecidos en sectores críticos del municipio" vs. "accidentalidad total histórica del municipio").

In [5]:
agg_sectores = (
    df_sectores.groupby(['divipola', 'municipio', 'departamento'], as_index=False)['fallecidos']
    .sum()
    .rename(columns={'fallecidos': 'fallecidos_sectores_criticos', 'divipola': 'CodigoMunicipioHecho'})
)
print("Filas agregadas (sectores críticos, por municipio):", agg_sectores.shape[0])
agg_sectores.sort_values('fallecidos_sectores_criticos', ascending=False).head()

Filas agregadas (sectores críticos, por municipio): 133


,CodigoMunicipioHecho,municipio,departamento,fallecidos_sectores_criticos
94,52835,TUMACO,NARIÑO,134
2,5088,BELLO,ANTIOQUIA,105
125,76520,PALMIRA,VALLE DEL CAUCA,97
117,76111,BUGA,VALLE DEL CAUCA,92
53,25286,FUNZA,CUNDINAMARCA,87


In [6]:
agg_historico = (
    df_historico.groupby(['CodigoMunicipioHecho'], as_index=False)['ConteoRegistros']
    .sum()
    .rename(columns={'ConteoRegistros': 'accidentes_historico_2015_2023'})
)
print("Filas agregadas (histórico 2015-2023, por municipio):", agg_historico.shape[0])
agg_historico.head()

Filas agregadas (histórico 2015-2023, por municipio): 1044


,CodigoMunicipioHecho,accidentes_historico_2015_2023
0,-1,20
1,5001,2255
2,5002,15
3,5004,1
4,5030,52


## 4. Cruce de las fuentes (`pd.merge`) con verificación de filas

Cruce por `CodigoMunicipioHecho` (código DANE). Se usa `how='left'` desde sectores críticos, porque la Fuente 1 es la unidad de decisión del proyecto (los 316 tramos / 133 municipios con sectores críticos) — lo que interesa es enriquecerlos con su comportamiento histórico, no al revés.

In [7]:
print("Filas antes del cruce:")
print("  agg_sectores  :", agg_sectores.shape[0])
print("  agg_historico :", agg_historico.shape[0])

check = pd.merge(agg_sectores, agg_historico, on='CodigoMunicipioHecho', how='left', indicator=True)
sin_match = (check['_merge'] == 'left_only').sum()
print(f"\nMunicipios con sector crítico SIN match en el histórico: {sin_match} de {len(agg_sectores)}")

Filas antes del cruce:
  agg_sectores  : 133
  agg_historico : 1044

Municipios con sector crítico SIN match en el histórico: 0 de 133


In [8]:
df_cruzado = pd.merge(agg_sectores, agg_historico, on='CodigoMunicipioHecho', how='left')

print("Filas después del cruce:", df_cruzado.shape[0])
print("Columnas:", list(df_cruzado.columns))
df_cruzado.sort_values('fallecidos_sectores_criticos', ascending=False).head(10)

Filas después del cruce: 133
Columnas: ['CodigoMunicipioHecho', 'municipio', 'departamento', 'fallecidos_sectores_criticos', 'accidentes_historico_2015_2023']


,CodigoMunicipioHecho,municipio,departamento,fallecidos_sectores_criticos,accidentes_historico_2015_2023
94,52835,TUMACO,NARIÑO,134,476
2,5088,BELLO,ANTIOQUIA,105,465
125,76520,PALMIRA,VALLE DEL CAUCA,97,802
117,76111,BUGA,VALLE DEL CAUCA,92,326
53,25286,FUNZA,CUNDINAMARCA,87,180
114,76001,CALI,VALLE DEL CAUCA,80,3216
54,25290,FUSAGASUGÁ,CUNDINAMARCA,69,282
8,5837,TURBO,ANTIOQUIA,66,334
0,5045,APARTADÓ,ANTIOQUIA,59,171
128,76892,YUMBO,VALLE DEL CAUCA,59,266


**Verificación:** el cruce dio 133 filas — exactamente los 133 municipios con sector crítico, sin duplicarse ni perderse ninguno (`sin_match = 0`). Es un `left join` limpio: cada municipio con sector crítico queda enriquecido con su accidentalidad histórica total 2015-2023.

## 5. Diccionario de datos (dataset ya cruzado: `df_cruzado`)

| Columna | Tipo de dato | Significado | Fuente |
|---|---|---|---|
| `CodigoMunicipioHecho` | int64 | Código DIVIPOLA (DANE) del municipio. | Ambas fuentes (clave de cruce) |
| `municipio` | string | Nombre del municipio (en mayúsculas). | `rs3u-8r4q.csv` (Fuente 1) |
| `departamento` | string | Nombre del departamento. | `rs3u-8r4q.csv` (Fuente 1) |
| `fallecidos_sectores_criticos` | int64 | Suma de personas fallecidas en los tramos/sectores críticos identificados dentro de ese municipio (un municipio puede tener varios tramos). | `rs3u-8r4q.csv` (Fuente 1) |
| `accidentes_historico_2015_2023` | float64 (puede tener NaN) | Total de registros de accidentalidad reportados para ese municipio en el consolidado histórico del Observatorio ANSV, años cerrados 2015-2023 (suma de `ConteoRegistros`). `NaN` si el municipio no aparece en el histórico filtrado. | `02_Comportamiento_datos_históricos.csv` (Fuente 2) |

**Columnas originales relevantes, para referencia:**

| Columna | Tipo de dato | Significado | Fuente |
|---|---|---|---|
| `id_mt` | string | Identificador del tramo vial (puede venir `<Null>`). | Fuente 1 |
| `entidad` | string | Entidad responsable del tramo (ANI, INVIAS, etc.). | Fuente 1 |
| `tramo` | string | Nombre del corredor vial (ej. "Bogotá - Villavicencio"). | Fuente 1 |
| `gizscore` / `gipvalue` | float64 | Indicadores de siniestralidad/priorización del tramo (por verificar metodología exacta en la ficha técnica del dataset). | Fuente 1 |
| `latitud` / `longitud` | float64 | Coordenadas del punto crítico dentro del tramo. | Fuente 1 |
| `AnoHecho` | int64 | Año del hecho de tránsito (2015-2023 tras el filtro de alcance). | Fuente 2 |
| `Sexo`, `Edad`, `ActorVial`, `TipoVehiculoGrupo`, `ObjetoColision`, `CondicionVictima`, `DiaOcurrencia`, `Rango3horas` | varios | Variables de desagregación de cada registro de accidentalidad. | Fuente 2 |
| `ConteoRegistros` | int64 | Número de registros que representa esa fila agregada. | Fuente 2 |